# Maya v2 — continuous streaming with in-flight MFC commands

This notebook uses `maya_v2.py`, which keeps one persistent USB Serial
connection to the Teensy and runs the sensor-packet reader in a background
thread. That lets us switch MFCs *while* the sensor stream is running,
without stopping and restarting the stream.

**Safe to call during streaming:** `MFCControl.set_flow`, `MFCControl.blink`,
`PCF8575Control_.send_state` (binary command, no USB text response).

**NOT safe during streaming:** `MFCControl.read_registers` — the firmware
writes `DATA:...` on the USB Serial and would corrupt the binary packet
stream. `maya_v2.MFCControl` refuses it while streaming.

In [ ]:
import os
import time
import pickle
import numpy as np
import matplotlib.pyplot as plt

from maya_v2 import SerialCommunication, PCF8575Control_, MFCControl

In [ ]:
# --- Serial / hardware configuration ---
PORT = "/dev/ttyACM0"   # adjust to your Teensy's serial port
BAUDRATE = 115200
FREQUENCY = 1           # sampling frequency (Hz) set on the Teensy

# DAC configuration
dac1_value = 800
dac2_value = 200

# PCF8575 configuration
PCF_flags = [True] * 16
PCF_frequencies = ["0"] * 16

sensor_id = ["U1-TGS2602", "U2-TGS2610-D00", "U3-SP3S-AQ2", "U4-GSBT11-DXX",
             "U8-TGS2600", "U9-TGS2603", "U10-TGS2630", "U13-TGS2612-D00",
             "U14-TGS2620", "U15-MG-812", "U16-TGS-3830", "U19-TGS1820",
             "U20-TGS2611-E00", "U21-TGS2616-C00", "U22-WSP2110", "U25-TGS-3870",
             "U7-BME680"]

In [ ]:
# Open ONE persistent serial connection. All commands and the streaming
# reader share this same `SerialCommunication` object.
serial_comm = SerialCommunication(PORT, BAUDRATE)

# Push initial DAC + sampling-frequency config to the Teensy
serial_comm.send_message(f"{dac1_value},{dac2_value},{FREQUENCY}")

# Initial PCF8575 state (all sensors ON)
PCF_control = PCF8575Control_(serial_comm, flags=PCF_flags, frequencies=PCF_frequencies)
PCF_control.send_state()

# Zero all MFCs before we begin
MFC_control = MFCControl(serial_comm)
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)

## Experiment — gas sequence with uninterrupted streaming

We start one long recording, then, while the Teensy keeps streaming, we
issue MFC setpoint changes at each step. Every MFC switch is logged with a
timestamp in `times_seq` so the sequence can be aligned with the sensor
trace afterwards.

In [ ]:
def generate_sequence(n_reps=20, binary=False):
    if binary:
        sequence = n_reps * list(range(1, 7))
    else:
        sequence = n_reps * list(range(1, 4))
    np.random.shuffle(sequence)
    return sequence


def apply_class(mfc, cls, rate_on=1.0, rate_off=0.0):
    """Translate a class index to one or more MFC setpoints.

    1-3: single gas on MFC 1/2/3 at `rate_on`
    4-6: binary mixtures (1+2, 1+3, 2+3) at 50/50
    """
    if cls in (1, 2, 3):
        mfc.set_flow(cls, rate_on)
    elif cls == 4:
        mfc.set_flow(1, rate_on / 2); mfc.set_flow(2, rate_on / 2)
    elif cls == 5:
        mfc.set_flow(1, rate_on / 2); mfc.set_flow(3, rate_on / 2)
    elif cls == 6:
        mfc.set_flow(2, rate_on / 2); mfc.set_flow(3, rate_on / 2)
    else:
        raise ValueError(f"Unknown class {cls}")


def clear_class(mfc, cls, rate_off=0.0):
    if cls in (1, 2, 3):
        mfc.set_flow(cls, rate_off)
    elif cls == 4:
        mfc.set_flow(1, rate_off); mfc.set_flow(2, rate_off)
    elif cls == 5:
        mfc.set_flow(1, rate_off); mfc.set_flow(3, rate_off)
    elif cls == 6:
        mfc.set_flow(2, rate_off); mfc.set_flow(3, rate_off)

In [ ]:
# --- Experiment parameters ---
t_baseline = 60          # seconds of baseline before the sequence
t_sample   = 20          # seconds each gas class is on
t_flush    = 0           # optional flush time between classes (0 = no flush)
n_sequence = 150         # total stimuli (for singles: n_sequence / 3 reps/class)
binary     = False       # False: classes 1-3; True: classes 1-6
filename   = f"data/v2_{n_sequence}_{t_sample}.csv"

reps = int(n_sequence / (6 if binary else 3))
sequence = generate_sequence(reps, binary=binary)

if os.path.exists(filename):
    os.remove(filename)
os.makedirs(os.path.dirname(filename), exist_ok=True)

times_seq = []   # list of (timestamp, event) tuples

# --- Start streaming ONCE, for the whole experiment ---
serial_comm.start_streaming(filename=filename, save_csv=True)
try:
    t0 = time.time()
    times_seq.append((time.time(), time.asctime(), "baseline_start", 0))
    time.sleep(t_baseline)

    for cls in sequence:
        # Turn gas ON (stream keeps running)
        apply_class(MFC_control, cls, rate_on=1.0)
        times_seq.append((time.time(), time.asctime(), "on", int(cls)))
        time.sleep(t_sample)

        # Turn gas OFF
        clear_class(MFC_control, cls, rate_off=0.0)
        times_seq.append((time.time(), time.asctime(), "off", int(cls)))

        if t_flush > 0:
            time.sleep(t_flush)

    times_seq.append((time.time(), time.asctime(), "sequence_end", 0))
    print(f"Experiment wall-time: {time.time() - t0:.1f} s")
finally:
    # Always stop the stream, even if the cell is interrupted.
    serial_comm.stop_streaming()

with open(f"{filename}_sequence.pkl", "wb") as f:
    pickle.dump(times_seq, f)
print(f"Sequence log saved to {filename}_sequence.pkl")

## Cleanup

In [ ]:
# Zero the MFCs and close the serial port when you're done.
for addr in (1, 2, 3):
    MFC_control.set_flow(addr, 0.0)
serial_comm.close()

## Quick look at the recorded data

In [ ]:
import csv

sensor_data = []
times = []
with open(filename, "r") as f:
    reader = csv.reader(f)
    for row in reader:
        if not row or row[0] == "Timestamp":
            continue
        times.append(row[0])
        values = []
        for i in range(17):
            b1 = int(row[2 * i + 1]); b2 = int(row[2 * i + 2])
            values.append(int.from_bytes([b1, b2], byteorder="little"))
        sensor_data.append(values)
sensor_data = np.array(sensor_data)
print(f"Loaded {sensor_data.shape[0]} packets × {sensor_data.shape[1]} channels")

In [ ]:
if len(sensor_data) > 0:
    fig, ax = plt.subplots(4, 4, figsize=(16, 12), sharex=True)
    for i in range(4):
        for j in range(4):
            idx = i * 4 + j
            ax[i, j].plot(sensor_data[:, idx])
            ax[i, j].set_title(sensor_id[idx], fontsize=9)
            ax[i, j].set_xticks([])
    plt.tight_layout()
    plt.show()